In [5]:
import pandas as pd
from setfit import SetFitModel
import time

print("🚀 啟動模型實戰盲測 (大規模推論)")

# ==========================================
# 1. 載入模型與測試資料
# ==========================================
print("載入專屬 SetFit 模型...")
model = SetFitModel.from_pretrained("./my_intent_classifier_setfit")

print("載入 17,495 筆候選池資料...")
df_test = pd.read_csv('candidate_pool.csv')
df_test = df_test.dropna(subset=['name']).drop_duplicates(subset=['name']).reset_index(drop=True)

# ==========================================
# 2. 執行全量推論
# ==========================================
texts = df_test['name'].astype(str).tolist()

print(f"\n🔥 開始對 {len(texts)} 筆資料進行批次推論...")
start_time = time.time()

# 模型預測 (M3 Max 推論一萬多筆大約只需 5~15 秒)
preds = model.predict(texts)

print(f"✅ 推論完成！總耗時: {time.time() - start_time:.2f} 秒")

# ==========================================
# 3. 整理結果與比對
# ==========================================
df_test['setfit_predict'] = preds

# 如果這份 CSV 裡面有之前 Snorkel 留下的預測欄位，我們就拿來比對
if 'snorkel_label' in df_test.columns:
    # 找出 Snorkel 放棄投票 (ABSTAIN / NaN)，但 SetFit 勇敢給出答案的案例
    df_rescued = df_test[df_test['snorkel_label'].isna() | (df_test['snorkel_label'] == '放棄投票')]
    
    # 找出 Snorkel 有給答案，但 SetFit 決定「推翻」它的案例 (這通常是模型泛化能力的證明)
    df_diff = df_test[
        (df_test['snorkel_label'].notna()) & 
        (df_test['snorkel_label'] != '放棄投票') & 
        (df_test['snorkel_label'] != df_test['setfit_predict'])
    ]
else:
    df_rescued = pd.DataFrame()
    df_diff = pd.DataFrame()

🚀 啟動模型實戰盲測 (大規模推論)
載入專屬 SetFit 模型...


The tokenizer you are loading from './my_intent_classifier_setfit' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


載入 17,495 筆候選池資料...

🔥 開始對 6345 筆資料進行批次推論...
✅ 推論完成！總耗時: 2.90 秒


In [6]:
# ==========================================
# 4. 印出精彩案例
# ==========================================
print("\n" + "="*50)
print("🎯 隨機抽樣測試結果 (一般預測)")
print("="*50)
for _, row in df_test.sample(10, random_state=0).iterrows():
    print(f"[{row['setfit_predict']}] <- {row['name']}")

if not df_rescued.empty:
    print("\n" + "="*50)
    print(f"💡 模型顯靈：從 Snorkel 放棄的資料中，拯救並分類了 {len(df_rescued)} 筆！(隨機抽樣 5 筆)")
    print("="*50)
    for _, row in df_rescued.sample(min(5, len(df_rescued))).iterrows():
        print(f"文本: {row['name']}")
        print(f"預判: [{row['setfit_predict']}]\n")

if not df_diff.empty:
    print("\n" + "="*50)
    print(f"⚔️ 判決逆轉：SetFit 推翻了 Snorkel 規則的案例共有 {len(df_diff)} 筆！(隨機抽樣 5 筆)")
    print("="*50)
    for _, row in df_diff.sample(min(5, len(df_diff))).iterrows():
        print(f"文本: {row['name']}")
        print(f"原規則: [{row['snorkel_label']}] -> 新模型: [{row['setfit_predict']}]\n")

# 將所有測試結果存檔，方便您用 Excel 慢慢看
output_file = 'candidate_pool_with_setfit_predictions.csv'
df_test.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"\n💾 完整推論結果已儲存至: {output_file}")


🎯 隨機抽樣測試結果 (一般預測)
[借貸融資] <- 借錢 24H 全方位貸款公司
[黃色與特種行業] <- 台北酒店經紀 招募-活潑&氣質Girl  正職/兼職/寒暑打工
[借貸融資] <- Cash-資金周轉.融資貸款
[黃特種行業] <- 我們，經紀人
[博弈] <- GAME One娛樂城
[正常困難負樣本] <- 林森北路君悅酒店
[正常困難負樣本] <- JW Marriott Macau 澳門JW 萬豪酒店
[正常困難負樣本] <- 台中 林酒店 森林百匯
[借貸融資] <- 便利薪轉貸/大寮小額借款/大寮薪轉借款/大寮房屋二胎借款/
[借貸融資] <- 高雄當舖推薦【允富當舖】汽機車、金飾、鑽石、手錶｜借款貸款、收購典當

💾 完整推論結果已儲存至: candidate_pool_with_setfit_predictions.csv


In [7]:
import pandas as pd
import numpy as np
import re
from setfit import SetFitModel

# 1. 載入模型
model = SetFitModel.from_pretrained("./my_intent_classifier_setfit")

# 2. 準備測試文本
# 這裡我們加入蔣公，以及一些我們預期會中的句子
test_data = [
    "蔣公（Chiang Kai-shek）民族英雄 抗日領袖 時代巨人 自由燈塔",
    "專業快速撥款，免保人免留車，全台皆可辦",
    "台北萬豪酒店",
    "加入自救會，律師教你如何拿回被騙的錢",
    "今日天氣晴朗，適合去陽明山踏青", # 另一個 OOD 測試
    "台南信貸",
    "八大",
]

city_pattern = re.compile(r"(台北|台中|台南|高雄|新北|桃園|新竹|苗栗|彰化|南投|雲林|嘉義|屏東|宜蘭|花蓮|台東|澎湖|金門|連江|萬華|西門|板橋|中壢|五期|七期)")
def preprocess_texts(texts, replacement="[LOC]"):
    """
    處理字串清單，將地名替換為佔位符
    :param texts: List of strings (或是單一字串)
    :param replacement: 替換的文字，預設為 [LOC]
    :return: 處理後的 List of strings
    """
    # 如果傳入的是單一字串，轉成 list 處理再轉回去
    if isinstance(texts, str):
        return city_pattern.sub(replacement, texts)
    
    # 使用清單推導式進行高速替換
    return [city_pattern.sub(replacement, str(t)) for t in texts]

def smart_classify(raw_texts, threshold=0.65):
    texts = preprocess_texts(raw_texts)
    
    # 取得機率分佈
    probs = model.predict_proba(texts)
    # 取得預測標籤
    preds = model.predict(texts)
    
    results = []
    for i, text in enumerate(texts):
        max_prob = probs[i].max().item()
        label = preds[i]
        
        # 信心度判定邏輯
        if max_prob < threshold:
            final_label = "異常/未知 (OOD)"
        else:
            final_label = label
            
        results.append({
            "text": text,
            "prediction": final_label,
            "confidence": round(max_prob, 4),
            "original_thought": label # 就算信心低，也紀錄一下模型「原本猜什麼」
        })
    return pd.DataFrame(results)

# 執行分析
df_final = smart_classify(test_data)

# 顯示結果
print("📊 最終防禦系統推論結果：")
display(df_final)

The tokenizer you are loading from './my_intent_classifier_setfit' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


📊 最終防禦系統推論結果：


,text,prediction,confidence,original_thought
0,蔣公（Chiang Kai-shek）民族英雄 抗日領袖 時代巨人 自由燈塔,正常困難負樣本,0.9985,正常困難負樣本
1,專業快速撥款，免保人免留車，全台皆可辦,借貸融資,0.9993,借貸融資
2,[LOC]萬豪酒店,正常困難負樣本,0.9968,正常困難負樣本
3,加入自救會，律師教你如何拿回被騙的錢,異常/未知 (OOD),0.5376,借貸融資
4,今日天氣晴朗，適合去陽明山踏青,正常困難負樣本,0.9961,正常困難負樣本
5,[LOC]信貸,借貸融資,0.9996,借貸融資
6,八大,正常困難負樣本,0.9986,正常困難負樣本
